In [ ]:
import io
import sys
import tarfile
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve().parents[1] if Path.cwd().name == "docs" else (Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.common.parser import AIRRParser, VDJdbSlimParser, VDJtoolsParser
from mir.common.repertoire import LocusRepertoire
from mir.utils.notebook_assets import (
    ensure_airr_benchmark,
    find_airr_benchmark_sra_meta,
    find_airr_benchmark_vdjdb_slim,
)

benchmark_root = ensure_airr_benchmark(repo_root, allow_patterns=["sra/**", "vdjdb/**", "vdjtools_lite/**"])
vdjdb_path = find_airr_benchmark_vdjdb_slim(benchmark_root)
sra_tarball, sra_meta_path = find_airr_benchmark_sra_meta(benchmark_root)
vdjtools_candidates = sorted((benchmark_root / "vdjtools_lite").glob("*.txt.gz"))
if not vdjtools_candidates:
    raise FileNotFoundError(f"No VDJtools files found under {benchmark_root / 'vdjtools_lite'}")
vdjtools_path = vdjtools_candidates[0]

print(f"vdjdb_path = {vdjdb_path}")
print(f"sra_tarball = {sra_tarball}")
print(f"sra_meta_path = {sra_meta_path}")
print(f"vdjtools_path = {vdjtools_path}")

# Parsing AIRR, VDJtools, and VDJdb Formats

In [ ]:
sample = VDJdbSlimParser().parse_file(vdjdb_path, species="HomoSapiens")
rep = sample["TRA"] if "TRA" in sample.loci else next(iter(sample.loci.values()))
print([(c.v_gene, c.junction_aa) for c in rep[:5]])
print(rep)

TypeError: VDJdbSlimParser() takes no arguments

Parse AIRR rearrangement files from the AIRR benchmark SRA tarball

The benchmark SRA bundle stores AIRR columns without an explicit `locus` field,
so this example tags the selected file as `TRB` before handing it to `AIRRParser`.

In [ ]:
with tarfile.open(sra_tarball, "r:gz") as tar:
    members = sorted(name for name in tar.getnames() if name.endswith(".tsv"))
    if not members:
        raise FileNotFoundError(f"No AIRR TSV members found in {sra_tarball}")
    target_member = members[0]
    with tar.extractfile(target_member) as raw:
        airr_df = pd.read_csv(io.BytesIO(raw.read()), sep="\t")

airr_df["locus"] = "TRB"
clonotypes = AIRRParser(locus="TRB").parse_inner(airr_df)
rep = LocusRepertoire(clonotypes=clonotypes, locus="TRB")
print(target_member)
print([(c.v_gene, c.junction_aa) for c in rep[:5]])
print(rep)

[(Unknown TRBV5-1*01:-1:?, 'CASSWGKGRGLRTDTQYF'), (Unknown TRBV19*01:-1:?, 'CASSIIVRGIQNTEAFF'), (Unknown TRBV27*01:-1:?, 'CASSLAWGPRNQPQHF'), (Unknown TRBV5-1*01:-1:?, 'CASSLARGAYEQYF'), (Unknown TRBV5-1*01:-1:?, 'CASSQAVLYEKLFF')]
Repertoire of 100000 clonotypes and 100000 cells:
κ0 CASSWGKGRGLRTDTQYF TGCGCCAGCAGCTGGGGAAAGGGGAGGGGCCTCCGCACAGATACGCAGTATTTT
κ1 CASSIIVRGIQNTEAFF TGTGCCAGTAGTATTATCGTCAGGGGGATTCAGAACACTGAAGCTTTCTTT
κ2 CASSLAWGPRNQPQHF TGTGCCAGCAGTTTAGCTTGGGGACCCCGCAATCAGCCCCAGCATTTT
κ3 CASSLARGAYEQYF TGCGCCAGCAGCTTGGCTCGGGGGGCCTACGAGCAGTACTTC
κ4 CASSQAVLYEKLFF TGCGCCAGCAGCCAGGCGGTACTTTATGAAAAACTGTTTTTT
{'path': 'assets/olga_humanTRB.txt'}
...


Load VDJtools format data from `airr_benchmark/vdjtools_lite`

In [ ]:
parser = VDJtoolsParser()
rep = LocusRepertoire(
    clonotypes=parser.parse(str(vdjtools_path)),
    locus="TRB",
)
print([(c.duplicate_count, c.junction_aa) for c in rep[:5]])
print(rep)

[(33675, 'CASRSTGFYNEQFF'), (33660, 'CASSRTGNEQYF'), (27294, 'CAISESLGEQFF'), (18291, 'CASSFEQGNSWTQYF'), (16343, 'CASSLGADTQYF')]
Repertoire of 137624 clonotypes and 517858 cells:
κ0 CASRSTGFYNEQFF TGTGCCAGCAGATCAACAGGGTTCTACAATGAGCAGTTCTTC
κ1 CASSRTGNEQYF TGTGCCAGCAGCAGGACAGGGAACGAGCAGTACTTC
κ2 CAISESLGEQFF TGTGCCATCAGTGAGTCGCTTGGTGAGCAGTTCTTC
κ3 CASSFEQGNSWTQYF TGTGCCAGCAGTTTCGAGCAGGGAAATTCGTGGACCCAGTACTTC
κ4 CASSLGADTQYF TGTGCCAGCAGCTTAGGGGCAGATACGCAGTATTTT
{'path': 'assets/samples/aging_3year/A5-S23.txt.gz'}
...
